In [11]:
%matplotlib
import torch
import numpy as np
import matplotlib.pyplot as plt

Using matplotlib backend: TkAgg


In [2]:
class BezierCurveAction():

    def __init__(self,
                 t_th_min,
                 t_th_max,
                 x_theta_min,
                 x_theta_max,
                 x_r_min,
                 x_r_max,
                 xd_theta_min,
                 xd_theta_max,
                 xd_r_min,
                 xd_r_max,
                 psi_min,
                 psi_max,
                 theta_min,
                 theta_max,
                 phi_min,
                 phi_max,
                 psid_min,
                 psid_max,
                 thetad_min,
                 thetad_max,
                 phid_min,
                 phid_max) -> None:

        self.t_th_min = t_th_min
        self.t_th_max = t_th_max

        self.x_theta_min = x_theta_min
        self.x_theta_max = x_theta_max

        self.x_r_min = x_r_min
        self.x_r_max = x_r_max

        self.xd_theta_min = xd_theta_min
        self.xd_theta_max = xd_theta_max

        self.xd_r_min = xd_r_min
        self.xd_r_max = xd_r_max

        self.psi_min = psi_min
        self.psi_max = psi_max

        self.theta_min = theta_min
        self.theta_max = theta_max

        self.phi_min = phi_min
        self.phi_max = phi_max

        self.psid_min = psid_min
        self.psid_max = psid_max

        self.thetad_min = thetad_min
        self.thetad_max = thetad_max

        self.phid_min = phid_min
        self.phid_max = phid_max

    def torch_cart2sph(self, pos: torch.Tensor):
        # Extract x, y, z components
        x = pos[:, 0]
        y = pos[:, 1]
        z = pos[:, 2]

        # Compute spherical coordinates
        hxy = torch.hypot(x, y)
        r = torch.hypot(hxy, z)
        el = torch.atan2(z, hxy)
        az = torch.atan2(y, x)

        # Concatenate azimuth, elevation, and radius
        spherical = torch.stack((az, el, r), dim=1)

        return spherical

    def torch_sph2cart(self, pos: torch.Tensor):
        # Extract az, el, r components
        az = pos[:, 0]
        el = pos[:, 1]
        r = pos[:, 2]

        rcos_theta = r * torch.cos(el)
        x = rcos_theta * torch.cos(az)
        y = rcos_theta * torch.sin(az)
        z = r * torch.sin(el)

        # Concatenate x, y, and z
        cartesian = torch.stack((x, y, z), dim=1)

        return cartesian

    def compute_bezier_w(self, start: torch.Tensor, start_v: torch.Tensor, end: torch.Tensor, end_v: torch.Tensor, t_th: torch.Tensor):

        w = torch.stack((
            start.unsqueeze(1),
            (((t_th / 3) * start_v) + start).unsqueeze(1),
            ((-(t_th / 3) * end_v) + end).unsqueeze(1),
            end.unsqueeze(1)), dim=2).squeeze()

        w = w.reshape(-1, 4, 3)

        w_d = torch.stack((
            ((3 / t_th) * (w[:, 1] - w[:, 0])).unsqueeze(1),
            ((3 / t_th) * (w[:, 2] - w[:, 1])).unsqueeze(1),
            ((3 / t_th) * (w[:, 3] - w[:, 2])).unsqueeze(1)), dim=2).squeeze()

        return w, w_d

    def bezier_trajectory(self, w: torch.Tensor, w_d: torch.Tensor, t_ex: float, t_th: torch.Tensor):

        t = t_ex / t_th

        bezier_curve_3 = torch.cat((
            (1.0) * (t**0) * (1 - t)**(3 - 0),
            (3.0) * (t**1) * (1 - t)**(3 - 1),
            (3.0) * (t**2) * (1 - t)**(3 - 2),
            (1.0) * (t**3) * (1 - t)**(3 - 3),
        ), dim=-1)

        bezier_curve_2 = torch.cat((
            (1.0) * (t**0) * (1 - t)**(2 - 0),
            (2.0) * (t**1) * (1 - t)**(2 - 1),
            (1.0) * (t**2) * (1 - t)**(2 - 2),
        ), dim=-1)

        bezier_position = torch.cat((
            (w[..., 0] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
            (w[..., 1] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
            (w[..., 2] * bezier_curve_3).sum(dim=-1).reshape(-1, 1)), dim=1)

        bezier_velocity = torch.cat((
            (w_d[..., 0] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
            (w_d[..., 1] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
            (w_d[..., 2] * bezier_curve_2).sum(dim=-1).reshape(-1, 1)), dim=1)

        return bezier_position, bezier_velocity

    def process_actions(self, actions: torch.Tensor, target: torch.Tensor):

        trunk_x_0 = torch.tensor([[0., 0., 0.3]])
        trunk_xd_0 = torch.tensor([[0., 0., 0.]])

        trunk_tg = trunk_x_0 + target

        self.t_th = (self.t_th_max - self.t_th_min) * 0.5 * (actions[..., 0] + 1) + self.t_th_min
        self.t_th = self.t_th.reshape(-1, 1)

        # Phi is the same for x and xd
        x_xd_phi = self.torch_cart2sph(trunk_tg)[..., 0].clone()

        # Calculate X_lo
        x_theta = (self.x_theta_max - self.x_theta_min) * 0.5 * (actions[..., 1] + 1) + self.x_theta_min
        x_r = (self.x_r_max - self.x_r_min) * 0.5 * (actions[..., 2] + 1) + self.x_r_min

        trunk_x_lo = self.torch_sph2cart(torch.stack((x_xd_phi, x_theta, x_r), dim=1))

        # Calculate Xd_lo

        xd_theta = (self.xd_theta_max - self.xd_theta_min) * 0.5 * (actions[..., 3] + 1) + self.xd_theta_min
        xd_r = (self.xd_r_max - self.xd_r_min) * 0.5 * (actions[..., 4] + 1) + self.xd_r_min

        trunk_xd_lo = self.torch_sph2cart(torch.stack((x_xd_phi, xd_theta, xd_r), dim=1))

        self.w_x, self.w_xd = self.compute_bezier_w(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, self.t_th)

        print(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, self.t_th)

    def eval_bezier(self, dt):

        x, xd = self.bezier_trajectory(self.w_x, self.w_xd, dt, self.t_th)

        return x, xd

In [3]:
bezierAction = BezierCurveAction(t_th_min=0.1,
                                 t_th_max=1,
                                 x_theta_min=np.pi / 4,
                                 x_theta_max=np.pi / 2,
                                 x_r_min=0.2,
                                 x_r_max=0.4,
                                 xd_theta_min=np.pi / 6,
                                 xd_theta_max=np.pi / 2,
                                 xd_r_min=0.1,
                                 xd_r_max=5,
                                 psi_min=-2 * np.pi,
                                 psi_max=2 * np.pi,
                                 theta_min=- 2 * np.pi,
                                 theta_max=2 * np.pi,
                                 phi_min=- 2 * np.pi,
                                 phi_max=2 * np.pi,
                                 psid_min=-4,
                                 psid_max=4,
                                 thetad_min=-4,
                                 thetad_max=4,
                                 phid_min=-4,
                                 phid_max=4,)

In [4]:
target = torch.tensor([[0.5, 0, 0]])

In [91]:
# action = torch.tensor([[ 0.3028,  0.4664, -0.0724,  0.1421, -0.1344]])
action = torch.tensor([[0.5, 0, 1, -1, -0.15]])
# action = torch.tensor([[0.5, 0, 1, -0.2, -0.2]])

In [92]:
bezierAction.process_actions(action, target)

tensor([[0.0000, 0.0000, 0.3000]]) tensor([[0., 0., 0.]]) tensor([[0.1531, 0.0000, 0.3696]]) tensor([[1.8901, 0.0000, 1.0912]]) tensor([[0.7750]])


In [93]:
t_th = bezierAction.t_th.item()
t_th

0.7749999761581421

In [94]:
trajectory = [bezierAction.eval_bezier(i) for i in np.arange(0,t_th,0.005)]
pos = torch.stack([val[0] for val in trajectory], dim=1)
lin_vel = torch.stack([val[1] for val in trajectory], dim=1)

In [95]:
plt.close('all')

In [96]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.35)
ax.set_xlim(-0.35, 0.35)
ax.set_ylim(-0.35, 0.35)
ax.scatter(pos[..., 0], pos[..., 1], pos[..., 2], color='blue')
ax.scatter(bezierAction.w_x[..., 0], bezierAction.w_x[..., 1], bezierAction.w_x[..., 2], color='purple', linewidths=5)
ax.quiver(pos[..., 0], pos[..., 1], pos[..., 2], lin_vel[..., 0], lin_vel[..., 1], lin_vel[..., 2], length=0.1, normalize=False, color='red', arrow_length_ratio=0.1)
plt.show()

In [97]:
fig, ax = plt.subplots(4, 1, figsize=(10, 8))
ax[0].plot(np.arange(0,t_th,0.005),pos[...,0].flatten())
ax[0].set_ylabel("X")
ax[1].plot(np.arange(0,t_th,0.005),pos[...,1].flatten())
ax[1].set_ylabel("Y")
ax[2].plot(np.arange(0,t_th,0.005),pos[...,2].flatten())
ax[2].set_ylabel("Z")
ax[2].axhline(0.15,color='red')
# vel magnitude
ax[3].plot(np.arange(0,t_th,0.005),np.linalg.norm(lin_vel, axis=-1).flatten())
ax[3].set_ylabel("vel")
plt.show()